# Grad-CAM 可视化

In [ ]:
import sys; sys.path.append("..")
import os, torch, matplotlib.pyplot as plt
from src.config import Config
cfg = Config(); device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from src.data.dataset import OralFluorescenceDataset
from src.data.transforms import get_val_transforms
from torch.utils.data import DataLoader
tf = get_val_transforms()
ds = OralFluorescenceDataset(cfg.data.data_root, os.path.join(cfg.data.split_dir, cfg.data.test_split), transform=tf, class_names=cfg.data.class_names)
loader = DataLoader(ds, 1, shuffle=True, num_workers=0)

In [ ]:
from src.models.backbone import create_model
from src.interpret.gradcam import find_target_layer, visualize_gradcam
model = create_model(cfg)
ckpt = "checkpoints/baseline_v1/best_model.pth"
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device)["model_state_dict"])
    model.to(device)
target_layer = find_target_layer(model, cfg.model.backbone)

In [ ]:
# 生成Grad-CAM
for i, (img, lbl) in enumerate(loader):
    if i >= 5: break
    fig = visualize_gradcam(model, img[0], device, cfg.data.class_names, target_layer)
    plt.show()